# Run Your Training as Anyscale Jobs
© 2026, Anyscale. All Rights Reserved

Notebooks 01 to 03 ran your training loop, Ray Train and Ray Tune interactively, inside a workspace kernel you were watching. This notebook moves the same scripts out of that kernel and into Anyscale Jobs: unattended runs that start their own cluster, run an entrypoint once, report a terminal state, and tear the cluster down. Nobody has to keep a browser tab open for a job to finish.

<div class="alert alert-block alert-info">

<b> Here is the roadmap for this notebook </b>

<ol>
  <li>Why jobs</li>
  <li>Read the job config</li>
  <li>Submit from inside the workspace</li>
  <li>When a job fails before it starts</li>
  <li>Submit from your laptop</li>
  <li>Submit the search as a job</li>
  <li>Find the outputs</li>
  <li>Activity: watch a job resume</li>
  <li>Next steps</li>
</ol>

</div>

**Imports**

In [ ]:
import pyarrow.fs
import torch
from ray.train import Checkpoint

from src.settings import resolve_storage_path

## 1. Why jobs

A workspace and a job solve different problems. Use a workspace while you are still writing and debugging code; use a job once that code is proven and needs to run without you.

| | Workspace | Job |
|---|---|---|
| Lifetime | Stays up until you stop it or it idles out | Starts a cluster, runs the entrypoint once, tears the cluster down |
| Failure handling | You watch the output and rerun cells by hand | `max_retries` restarts the whole job automatically |
| Environment | Whatever you built up interactively: installed packages, `git pull`, edited files | Exactly the image and directory named in the job config, uploaded fresh on every submission |
| Where outputs go | Wherever you pointed them, often `/mnt/cluster_storage`, which disappears with the workspace | Artifact storage by convention in this repo, so results outlive the cluster that produced them |
| Who starts it | You, interactively | You, a CI pipeline, a schedule, or another job |

Notebooks 01 to 03 lived in the left column. Everything below moves `src/train_torch.py`, `src/train_ray_train.py` and `src/tune_ray_train.py` into the right one, unchanged.

## 2. Read the job config

The three job configs in `jobs/` wrap the three scripts you already ran by hand. Start with the one that trains distributed:

In [ ]:
!cat jobs/job_02_ray_train.yaml

| Field | What it does |
|---|---|
| `name` | The job's display name in the console and CLI; independent of any training run name inside the script |
| `entrypoint` | The command Anyscale runs on the head node once the cluster is up |
| `working_dir` | Set to `.`; combined with `--working-dir .` on the CLI, this is the directory Anyscale uploads |
| `excludes` | Skips `.git`, notebooks, data and caches from that upload |
| `image_uri` | The container image every node in the job's cluster runs |
| `compute_config` | The registered compute shape: a CPU head plus GPU worker groups |
| `env_vars` | Environment variables `Settings.from_env()` reads inside the job (`NUM_WORKERS`, `USE_GPU`, `NUM_EPOCHS`, `STORAGE_PATH`, `RAY_TRAIN_WORKER_GROUP_START_TIMEOUT_S`) |
| `max_retries` | How many times Anyscale restarts the whole job if the driver process dies |

<div class="alert alert-block alert-warning">

<b>Why the worker-group timeout is 1800, not the 60 default</b>

Ray Train v2's default worker group startup timeout assumes the workers it is waiting for already exist. `gpu-multinode-dev` scales its GPU worker groups from zero, so provisioning the first worker can take minutes on its own, longer still if the autoscaler has to fall back to another zone or instance type. Without `RAY_TRAIN_WORKER_GROUP_START_TIMEOUT_S: "1800"`, `job_02` and `job_03` would fail about a minute in with `WorkerGroupStartupTimeoutError`, before training ever starts. `job_01` doesn't need this variable because it uses a plain Ray task, not a Ray Train worker group.

</div>

<div class="alert alert-block alert-info">

<b>Why the workers can already <code>import src</code></b>

`anyscale job submit --working-dir .` uploads this directory and sets `working_dir` on the **job's** runtime env, not just the driver's. Every Ray worker the job creates inherits that runtime env, so `from src.settings import Settings` works on a worker with no extra help from you.

A workspace has no job-level runtime env to inherit from, which is why `src/settings.py` defines `ray_init_with_repo()` instead of calling `ray.init()` directly: it first tries shipping the repo as a `runtime_env` argument to `ray.init()`, the way a workspace needs, and falls back to a bare `ray.init()` the moment Ray reports that a runtime env's `working_dir` is already set, which is exactly the conflict a job produces. Every script in this repo calls `ray_init_with_repo()` for this reason, and never `ray.init()` on its own.

</div>

<div class="alert alert-block alert-info">

<b>Two different retry knobs</b>

`max_retries` in the job config is Anyscale deciding whether to restart the **entire job**: a new cluster, a new driver process, the entrypoint from the top. `FailureConfig(max_failures=...)` inside `src/train_ray_train.py` is Ray Train deciding whether to restart just the **worker group**, while the same driver process keeps running. Both are worth setting on preemptible capacity: a single preempted worker shouldn't cost you a whole new cluster, and a dead driver shouldn't end the run either.

By default, a job restarted by `max_retries` calls `main()` again from scratch, which picks a fresh timestamped run name and starts training over. Add `RUN_NAME` to `env_vars` (a fixed string, not the timestamped default) and a retried job resumes from the last checkpoint instead, because `build_trainer` reuses the same `RunConfig(name=..., storage_path=...)` on every attempt, and `load_checkpoint_state` finds whatever a previous attempt already wrote there. Section 8 has you try this directly.

</div>

## 3. Submit from inside the workspace

The commands below run from the repo root, inside this workspace, exactly as they would in a terminal. `job_02_ray_train.yaml` is the job you'd actually use to train the distributed script from notebook 02, but to keep this notebook's own run short, the one job it submits and waits on for real is `job_01_pytorch.yaml`, the fastest of the three. The identical command shape works unchanged for `job_02` and `job_03`, shown as commands to run yourself in sections 5 and 6.

In [ ]:
!anyscale job submit -f jobs/job_01_pytorch.yaml --working-dir . --name nb04-pytorch --wait

`--wait` blocks and streams logs until the job reaches a terminal state. Expect a few minutes: most of it is `gpu-multinode-dev` scaling a GPU worker up from zero, not the training itself.

In [ ]:
!anyscale job status -n nb04-pytorch

In [ ]:
!anyscale job logs -n nb04-pytorch --tail --max-lines 50

Every job also gets a page in the console with Logs, Metrics, and, for a job that runs a distributed Ray Train trainer, a Ray Train tab showing per-worker progress live. `job_01` is a single Ray task, not a Ray Train run, so its Ray Train tab has nothing to show; that tab is worth a look once you submit `job_02` yourself in section 5.

## 4. When a job fails before it starts

Not every `FAILED` job looks the same, and the two shapes call for different responses.

One shape is your entrypoint raising an exception after training started: the logs endpoint has application output, and `anyscale job logs` shows a traceback pointing at a line in `src/`. That's a code problem, and the log is where you'd expect to look.

The other shape is a job that never got a cluster at all: `FAILED`, an empty `runs` list, and the logs endpoint returning 404. There is nothing to show because nothing ran; the code you wrote was never the problem. This happened for real during this repo's own development: a CPU quota blocked every head node `gpu-multinode-dev` tried, across four availability zones, so the job never started. Resource quotas and Kubernetes scheduling rules (see [docs/kubernetes-clouds.md](../docs/kubernetes-clouds.md) for the Kubernetes case) both produce this identical shape.

The answer lives in cluster lifecycle events, not application logs:

```text
anyscale job status --id <job-id> -v
```
```text
{
  "id": "prodjob_abc123",
  "name": "nb04-pytorch",
  "state": "FAILED",
  "runs": []
}
```

An empty `runs` list is the tell: no run ever started, so there are no application logs to fetch. The console job page's Events tab is the other place to look, recording what the autoscaler tried and why each attempt was rejected, before any application code had a chance to run.

## 5. Submit from your laptop

The exact same command works unchanged from a laptop with the Anyscale CLI installed and logged in, no workspace involved:

```text
anyscale job submit -f jobs/job_02_ray_train.yaml --working-dir . --name nb04-train --wait
```

Two things make this work with nothing else set up on the laptop.

First, `compute_config: gpu-multinode-dev` and `image_uri: anyscale/image/pytorch-to-ray-train:1` are both named directly in the job config. Inside a workspace, a job config that omits either field can inherit it from the workspace it was submitted from; from a laptop there is no workspace to inherit from, so both need to be explicit. This repo's job configs always name them, which is exactly why the same file works in both places.

Second, `STORAGE_PATH: "artifact://pytorch-to-ray-train/jobs"` in `env_vars` is a sentinel, not a real path. `resolve_storage_path()` in `src/settings.py` resolves it against `$ANYSCALE_ARTIFACT_STORAGE`, and that resolution happens inside the job, on the cluster Anyscale creates, where that variable is always set. Your laptop never needs to know it, or anything else about the Anyscale environment: the CLI's own job is only to upload the working directory and ask for a cluster; every environment-specific decision happens after the cluster exists.

## 6. Submit the search as a job

The hyperparameter sweep from notebook 03 submits the same way:

In [ ]:
!cat jobs/job_03_tune.yaml

`NUM_SAMPLES` and `MAX_CONCURRENT_TRIALS` are the same resource-math inputs from notebook 03: peak GPU demand is their product times `NUM_WORKERS`, which the comment in the config works out to 4.

```text
anyscale job submit -f jobs/job_03_tune.yaml --working-dir . --name nb04-tune --wait
```

## 7. Find the outputs

`job_01`, and any of `job_02` or `job_03` you ran yourself above, all write checkpoints under the same artifact storage prefix, using the identical `STORAGE_PATH` sentinel. None of those checkpoints depend on the job's cluster still existing. Prove that from this workspace's own kernel, using nothing but `pyarrow.fs`.

In [ ]:
prefix = resolve_storage_path("artifact://pytorch-to-ray-train/jobs")
print("artifact storage prefix:", prefix)

fs, root = pyarrow.fs.FileSystem.from_uri(prefix)
top_level = sorted(
    info.base_name
    for info in fs.get_file_info(pyarrow.fs.FileSelector(root, recursive=False))
    if info.type == pyarrow.fs.FileType.Directory
)
print(f"{len(top_level)} entries directly under the prefix:")
for name in top_level:
    print(" ", name)

In [ ]:
def find_latest_checkpoint_dir(fs: pyarrow.fs.FileSystem, root: str) -> str:
    """Every checkpoint any script in this repo writes contains a model.pt.

    Walk the whole prefix recursively and return the directory holding the
    most recently written one, regardless of which job wrote it or how
    deeply Ray Train nested it under its own run directory.
    """
    candidates = [
        info
        for info in fs.get_file_info(pyarrow.fs.FileSelector(root, recursive=True))
        if info.type == pyarrow.fs.FileType.File and info.base_name == "model.pt"
    ]
    if not candidates:
        raise RuntimeError(f"no model.pt found under {root}; run job_01 first")
    latest = max(candidates, key=lambda info: info.mtime)
    return latest.path.rsplit("/", 1)[0]


latest_dir = find_latest_checkpoint_dir(fs, root)
print("latest checkpoint directory:", latest_dir)

checkpoint = Checkpoint(path=latest_dir, filesystem=fs)
print("checkpoint:", checkpoint)

In [ ]:
with checkpoint.as_directory() as ckpt_dir:
    state_dict = torch.load(f"{ckpt_dir}/model.pt", map_location="cpu")

num_params = sum(p.numel() for p in state_dict.values())
print(f"loaded {len(state_dict)} tensors, {num_params:,} parameters, from a cluster that no longer exists")

That's the payoff of this whole notebook: a workspace, running right now, reading a checkpoint that a job's cluster wrote and then tore down, minutes or days ago, with no coordination between the two beyond a shared storage path.

## 8. Activity: watch a job resume

<div class="alert alert-block alert-info">

Using `jobs/job_02_ray_train.yaml`, submit it twice from a terminal (inside the workspace or from your laptop), setting `RUN_NAME` to the **same** fixed value both times and a **different** `NUM_EPOCHS`:

1. First: `RUN_NAME=resume-demo NUM_EPOCHS=2`
2. Second, after the first finishes: `RUN_NAME=resume-demo NUM_EPOCHS=4`

Read the second job's logs. Which epoch does it print first, and why?

</div>

In [ ]:
# Write your prediction here, then check it against the second job's logs.


<div class="alert alert-block alert-info">

<details>

<summary> Click here to see the solution </summary>

```text
anyscale job submit -f jobs/job_02_ray_train.yaml --working-dir . --name resume-1 \
  --env RUN_NAME=resume-demo --env NUM_EPOCHS=2 --wait
anyscale job submit -f jobs/job_02_ray_train.yaml --working-dir . --name resume-2 \
  --env RUN_NAME=resume-demo --env NUM_EPOCHS=4 --wait
anyscale job logs -n resume-2 --tail --max-lines 20
```

The second job's first rank-0 line reads `resuming from epoch 2`, not epoch 0. Both submissions used the same `RUN_NAME`, so `build_trainer` pointed both at the identical `RunConfig(name="resume-demo", storage_path=...)`, and `load_checkpoint_state` found the checkpoint the first job's last completed epoch left behind and picked up from there. Only epochs 2 and 3 run in the second job, not four epochs from scratch. If you instead gave the second submission a different `RUN_NAME`, it would start a brand-new run at epoch 0, writing to its own directory alongside the first.

</details>

</div>

## 9. Next steps

The CLI commands above are the fastest way to iterate, but two more pieces round out how jobs run in practice, without you submitting each one by hand:

- **Job queues** (`anyscale job-queue`) cap how many jobs run at once against a shared GPU budget, queueing the rest instead of letting them all compete for the same workers.
- **Schedules** (`anyscale schedule`) run a job config on a recurrence, useful for a training job that should retrain nightly on fresh data with nobody submitting it by hand.
- The same job configs submit equally well through the Anyscale Python SDK, which is how a tool like Airflow, or your platform team's own orchestrator, would trigger this repo's jobs as one step in a larger pipeline.

That closes the journey this repo set out to teach: an unchanged PyTorch loop placed on a GPU worker with one Ray task (notebook 01), the same loop rewritten for Ray Train v2 and made fault tolerant across GPU workers (notebook 02), a Ray Tune sweep over that same distributed trainer (notebook 03), and now every stage of it running unattended as an Anyscale Job, from a workspace or from a laptop, with checkpoints that outlive the cluster that wrote them (notebook 04). From here, the natural next step is your own model and data: the README's "Bring your own training script" section walks through exactly which pieces of `src/` to replace, and in what order.

## Further reading

| Topic | Link |
|---|---|
| Anyscale jobs | [docs.anyscale.com/jobs](https://docs.anyscale.com/jobs) |
| Jobs tutorial | [docs.anyscale.com/jobs/tutorial](https://docs.anyscale.com/jobs/tutorial) |
| Monitor jobs | [docs.anyscale.com/jobs/monitor](https://docs.anyscale.com/jobs/monitor) |
| Job queues | [docs.anyscale.com/jobs/queues](https://docs.anyscale.com/jobs/queues) |
| Schedules | [docs.anyscale.com/jobs/schedules](https://docs.anyscale.com/jobs/schedules) |
| CLI reference | [docs.anyscale.com/reference/cli](https://docs.anyscale.com/reference/cli) |
| SDK quickstart | [docs.anyscale.com/reference/quickstart-sdk](https://docs.anyscale.com/reference/quickstart-sdk) |
| Storage | [docs.anyscale.com/storage](https://docs.anyscale.com/storage) |
| Ray Train persistent storage | [docs.ray.io](https://docs.ray.io/en/latest/train/user-guides/persistent-storage.html) |
| Template: job-intro | [github.com/anyscale/templates](https://github.com/anyscale/templates/tree/main/templates/job-intro) |
| Template: anyscale-jobs-intro | [github.com/anyscale/templates](https://github.com/anyscale/templates/tree/main/templates/anyscale-jobs-intro) |
| Template: workspaces-dev-flow | [github.com/anyscale/templates](https://github.com/anyscale/templates/tree/main/templates/workspaces-dev-flow) |